# Programación orientada a objetos con Python
## Video 1. Del procedimiento al objeto

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Explicar qué problemas aparecen cuando los datos y las funciones que los usan viven separados.
2. Definir con precisión qué es un objeto: estado, comportamiento e identidad.
3. Reconocer el estado y el comportamiento de un objeto matemático.
4. Leer una primera clase sencilla, `Fraccion`, y compararla con el enfoque procedimental.

## 1. Planteamiento del problema

**Definición.** Una *fracción* es un par ordenado $(p, q)$ con $p, q \in \mathbb{Z}$ y $q \neq 0$.
Dos fracciones $(p, q)$ y $(r, s)$ son *equivalentes* cuando $p\,s = q\,r$.

**Problema: suma de fracciones.**

- **Entrada:** dos fracciones $(p, q)$ y $(r, s)$.
- **Salida:** la fracción irreducible $(a, b)$ con $b > 0$ que es equivalente a $(p\,s + q\,r,\; q\,s)$.

Primero lo resolvemos con lo que ya conocemos: tuplas y funciones.

## 2. Enfoque procedimental: datos por un lado, funciones por otro

In [ ]:
from math import gcd

def crear_fraccion(numerador, denominador):
    return (numerador, denominador)

def simplificar(f):
    p, q = f
    d = gcd(p, q)
    return (p // d, q // d)

def sumar(f, g):
    p, q = f
    r, s = g
    return simplificar((p * s + q * r, q * s))

def mostrar(f):
    return f"{f[0]}/{f[1]}"

In [ ]:
a = crear_fraccion(1, 2)
b = crear_fraccion(1, 3)
print(mostrar(sumar(a, b)))   # 1/2 + 1/3 = 5/6

Funciona. Pero observa qué **no** garantiza este diseño. Ejecuta las tres celdas siguientes.

### Falla 1. Una «fracción» inválida pasa sin avisar

In [ ]:
mala = crear_fraccion(1, 0)          # el denominador es cero: no es una fracción
print(mostrar(sumar(mala, a)))       # no hay error; el resultado no tiene sentido

### Falla 2. Representaciones no normalizadas

In [ ]:
c = crear_fraccion(1, -2)            # el signo quedó en el denominador
print(mostrar(c))                    # 1/-2, aunque pedimos denominador positivo

### Falla 3. Nada impide mezclar tipos de datos

In [ ]:
try:
    sumar((1, 2), (1, 2, 3))         # una tupla de tres elementos
except Exception as error:
    print(type(error).__name__, "->", error)

**Diagnóstico.** La regla «el denominador es distinto de cero y positivo» es una propiedad de los *datos*, pero está escrita (o, peor, **no** está escrita) en las *funciones*. Cada función tendría que volver a verificarla, y cualquiera puede crear una tupla inválida sin pasar por ninguna verificación.

## 3. Idea central: un objeto agrupa estado y comportamiento

**Definición.** Un *objeto* es una entidad que reúne:

1. **Estado:** los datos que lo describen (por ejemplo, numerador y denominador).
2. **Comportamiento:** las operaciones que puede realizar o que se pueden realizar sobre él (por ejemplo, sumar o mostrarse).
3. **Identidad:** lo que lo distingue de otro objeto, aunque tengan el mismo estado.

Una *clase* es la plantilla que describe cómo se construyen los objetos de un mismo tipo y qué operaciones tienen. Un objeto concreto construido con esa plantilla se llama *instancia*.

La ventaja que buscamos: **la regla sobre los datos vive junto a los datos**, en un solo lugar, y se verifica al crear el objeto.

## 4. Primer vistazo: la clase `Fraccion`

In [ ]:
class Fraccion:
    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        d = gcd(numerador, denominador)
        self.numerador = numerador // d
        self.denominador = denominador // d

    def sumar(self, otra):
        return Fraccion(
            self.numerador * otra.denominador + self.denominador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"

In [ ]:
x = Fraccion(1, 2)
y = Fraccion(1, 3)
print(x.sumar(y).mostrar())          # 5/6
print(Fraccion(1, -2).mostrar())     # -1/2: ya viene normalizada

In [ ]:
try:
    Fraccion(1, 0)
except ValueError as error:
    print("Se rechazó la fracción inválida:", error)

**Lo que acabamos de ganar** (en los próximos videos explicaremos cada pieza con detalle):

| Pregunta | Tuplas y funciones | Objeto `Fraccion` |
|---|---|---|
| ¿Dónde está la regla «denominador distinto de cero»? | En ningún lugar fijo | En `__init__`, una sola vez |
| ¿Se puede crear una fracción inválida? | Sí, sin aviso | No: se lanza un error |
| ¿Dónde viven las operaciones? | Sueltas en el programa | Dentro de la clase |
| ¿Cómo se llama una operación? | `sumar(x, y)` | `x.sumar(y)` |

**Estado de `x`:** `numerador = 1`, `denominador = 2`.
**Comportamiento de `x`:** `sumar` y `mostrar`.
**Identidad:** `x` y `Fraccion(1, 2)` creada otra vez son objetos distintos con el mismo estado.

In [ ]:
z = Fraccion(1, 2)
print(x is z)                        # False: dos objetos distintos
print(x.mostrar() == z.mostrar())    # True: el mismo estado

## 5. Ejercicios

**Ejercicio 1 (procedimental).** Escribe la función `multiplicar(f, g)` para fracciones representadas como tuplas.
- Entrada: dos tuplas $(p, q)$ y $(r, s)$.
- Salida: la tupla irreducible equivalente a $(p\,r,\; q\,s)$.

**Ejercicio 2 (objetos).** Agrega a la clase `Fraccion` el método `multiplicar(self, otra)`, que devuelva una nueva `Fraccion`.

**Ejercicio 3 (modelado).** Para cada objeto matemático, escribe su *estado* y dos operaciones de su *comportamiento*:
a) un vector en el plano; b) un polinomio; c) un intervalo $[a, b]$.

## 6. Soluciones

In [ ]:
# Ejercicio 1
def multiplicar(f, g):
    p, q = f
    r, s = g
    return simplificar((p * r, q * s))

print(mostrar(multiplicar((2, 3), (3, 4))))   # 1/2

In [ ]:
# Ejercicio 2
class Fraccion:
    def __init__(self, numerador, denominador):
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        d = gcd(numerador, denominador)
        self.numerador = numerador // d
        self.denominador = denominador // d

    def sumar(self, otra):
        return Fraccion(
            self.numerador * otra.denominador + self.denominador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def multiplicar(self, otra):
        return Fraccion(
            self.numerador * otra.numerador,
            self.denominador * otra.denominador,
        )

    def mostrar(self):
        return f"{self.numerador}/{self.denominador}"

print(Fraccion(2, 3).multiplicar(Fraccion(3, 4)).mostrar())   # 1/2

**Ejercicio 3.** Una respuesta posible:

| Objeto | Estado | Comportamiento |
|---|---|---|
| Vector en el plano | componentes $x$ y $y$ | sumar con otro vector; calcular su norma |
| Polinomio | lista de coeficientes | evaluar en un punto; derivar |
| Intervalo $[a, b]$ | extremos $a$ y $b$ con $a \le b$ | decidir si contiene un punto; calcular su longitud |

## 7. Resumen

- Cuando datos y funciones viven separados, las reglas sobre los datos se repiten o se olvidan.
- Un **objeto** reúne **estado**, **comportamiento** e **identidad**.
- Una **clase** es la plantilla; una **instancia** es un objeto concreto.
- En el siguiente video estudiaremos la clase con detalle: `class`, `__init__`, `self` y los atributos de instancia y de clase.